In [1]:

# %%
# ==============================================
# Week 4 Section 5 — Monitoring & Maintenance Planning (Business)
# Emits simple drift & performance checks and a monitoring template
# ==============================================

PROFILE = "dev"   # dev | preprod | final
SECTION_ID = "Wk04_Section5"

from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name.lower() == "code":
    BASE_DIR = BASE_DIR.parent
RESULTS_DIR = BASE_DIR / "results" / f"{SECTION_ID}_{PROFILE}"
REPORTS_DIR = RESULTS_DIR / "reports"
PLOTS_DIR   = RESULTS_DIR / "plots"
for d in (REPORTS_DIR, PLOTS_DIR): d.mkdir(parents=True, exist_ok=True)

CONSOLIDATED = BASE_DIR / "SDS-CP036-powercast_Wk04_Report_Business.md"
DATA_PATH   = BASE_DIR / "data" / "Tetuan City power consumption.csv"


# Common helpers
from pathlib import Path
import os, re, math, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Cap threads to keep runtime light
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Optional libraries
_HAVE_SM = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    _HAVE_SM = True
except Exception:
    _HAVE_SM = False

_HAVE_FBPROPHET = False
try:
    try:
        from prophet import Prophet
        _HAVE_FBPROPHET = True
    except Exception:
        from fbprophet import Prophet
        _HAVE_FBPROPHET = True
except Exception:
    _HAVE_FBPROPHET = False

_HAVE_XGB = False
try:
    from xgboost import XGBRegressor as _XGB
    def XGBRegressor(*args, **kwargs):
        kwargs.setdefault("n_estimators", 200)
        kwargs.setdefault("max_depth", 4)
        kwargs.setdefault("learning_rate", 0.1)
        kwargs.setdefault("subsample", 0.9)
        kwargs.setdefault("colsample_bytree", 0.9)
        kwargs.setdefault("tree_method", "hist")
        kwargs.setdefault("n_jobs", 1)
        return _XGB(*args, **kwargs)
    _HAVE_XGB = True
except Exception:
    _HAVE_XGB = False

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c.strip()) for c in df.columns]
    return df

def infer_step_and_freq(idx: pd.DatetimeIndex):
    try:
        freq = pd.infer_freq(idx[:1000])
    except Exception:
        freq = None
    if freq:
        step = pd.to_timedelta(pd.tseries.frequencies.to_offset(freq))
        return step, freq
    diffs = pd.Series(idx).diff().dropna()
    step = diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(hours=1)
    total_sec = int(step.total_seconds())
    if total_sec % 3600 == 0:
        freq = f"{total_sec // 3600}H"
    elif total_sec % 60 == 0:
        freq = f"{total_sec // 60}T"
    else:
        freq = f"{total_sec}S"
    return step, freq

def limit_last_days(df: pd.DataFrame, date_col="DateTime", max_days=None) -> pd.DataFrame:
    if not max_days:
        return df
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    cutoff = df[date_col].max() - pd.Timedelta(days=max_days)
    return df[df[date_col] >= cutoff]

def pick_zone_cols(df: pd.DataFrame, k: int):
    cols = [c for c in df.columns if re.match(r"Zone\s+\d+\s+Power\s+Consumption", c)]
    cols = sorted(cols, key=lambda s: int(re.search(r"\d+", s).group()))
    return cols[:k] if k else cols

def make_time_splits(df: pd.DataFrame, test_days: int, date_col="DateTime", steps_per_day: int = 24):
    df = df.sort_values(date_col)
    test_len = int(test_days * steps_per_day)
    if test_len >= len(df):
        test_len = max(1, len(df)//10)
    test = df.iloc[-test_len:].copy()
    train_val = df.iloc[:-test_len].copy()
    if len(train_val) > 100:
        cut = int(len(train_val) * 0.85)
        train, val = train_val.iloc[:cut].copy(), train_val.iloc[cut:].copy()
    else:
        train, val = train_val.copy(), pd.DataFrame(columns=train_val.columns)
    return train, val, test

def metrics_series(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.maximum(1e-8, np.where(y_true==0, 1e-8, y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    return dict(rmse=rmse, mae=mae, mape=mape)

def save_plot_ts(ts_df: pd.DataFrame, cols, title, path: Path):
    plt.figure(figsize=(10,4))
    for c in cols:
        plt.plot(ts_df.index, ts_df[c], label=c)
    plt.title(title); plt.xlabel("Time"); plt.ylabel("Power")
    plt.legend(loc="best"); plt.tight_layout()
    plt.savefig(path, bbox_inches="tight"); plt.close("all")

def rel(p: Path, base: Path) -> str:
    try: return p.relative_to(base).as_posix()
    except Exception: return p.as_posix()

def write_text(path: Path, text: str):
    path.write_text(text, encoding="utf-8")
    print("Wrote:", rel(path, Path.cwd()))


if not DATA_PATH.exists():
    write_text(REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section5_Business_Report.md",
               "# SDS-CP036-powercast – Week 4 Section 5: Monitoring & Maintenance Planning\n\n"
               f"Profile: `{PROFILE}`\n\n**Error:** data file not found: {rel(DATA_PATH, BASE_DIR)}")
    raise SystemExit(0)

df = pd.read_csv(DATA_PATH)
df = normalize_columns(df)
if "DateTime" not in df.columns:
    alts = [c for c in df.columns if c.lower()=="datetime"]
    if alts: df = df.rename(columns={alts[0]:"DateTime"})

zones = pick_zone_cols(df, 0)
df["DateTime"] = pd.to_datetime(df["DateTime"])
df = df.sort_values("DateTime")

# Simple drift check: compare last 7 days mean vs previous 30 days mean per zone
plan_rows = []
alerts = []
for z in zones:
    s = pd.to_numeric(df[z], errors="coerce")
    last_7 = s.iloc[-7*24*6:] if len(s)>0 else s  # assume 10-min → 6 per hour × 24 × 7
    prev_30 = s.iloc[-30*24*6-7*24*6:-7*24*6] if len(s)>0 else s
    mu7 = float(last_7.mean()) if len(last_7)>0 else np.nan
    mu30 = float(prev_30.mean()) if len(prev_30)>0 else np.nan
    drift = (mu7 - mu30) / (abs(mu30) + 1e-8) * 100.0 if np.isfinite(mu7) and np.isfinite(mu30) else np.nan
    plan_rows.append({"zone":z, "mean_7d":mu7, "mean_prev_30d":mu30, "pct_change":drift})
    if np.isfinite(drift) and abs(drift) > 15.0:
        alerts.append({"zone":z, "type":"data_drift", "detail":f"mean shifted {drift:.1f}% vs 30d"})

plan_csv = REPORTS_DIR / "monitoring_snapshot.csv"
pd.DataFrame(plan_rows).to_csv(plan_csv, index=False)
alerts_csv = REPORTS_DIR / "monitoring_alerts.csv"
pd.DataFrame(alerts).to_csv(alerts_csv, index=False)

# Create a monitoring playbook
playbook = REPORTS_DIR / "monitoring_playbook.md"
write_text(playbook, f"""
# Powercast — Monitoring Playbook (Week 4 Section 5)

**What to watch**  
- **Data health:** Missing timestamps, duplicate DateTime, outlier spikes.  
- **Feature drift:** 7‑day vs 30‑day mean/variance for demand and key drivers.  
- **Performance:** Weekly RMSE/MAE/MAPE vs. acceptance thresholds.

**Thresholds & Actions**  
- **Data drift:** |Δmean| > **15%** → open investigation; if sustained 2 weeks, retrain.  
- **Performance:** MAPE > **10%** for 2 consecutive weeks → tune or switch model.  
- **Anomalies:** Unexplained spikes → escalate to operations; annotate holiday/events.

**Who does what**  
- **Data engineering:** Pipeline checks & backfills.  
- **DS/ML:** Retraining, feature updates.  
- **Ops:** Calendar events, demand-side actions.

**Cadence**  
- **Daily:** Data health checks.  
- **Weekly:** Performance dashboard & drift review.  
- **Quarterly:** Architecture review and model refresh planning.
""")

s5_md = REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section5_Business_Report.md"
write_text(s5_md, f"""
# SDS-CP036-powercast – Week 4 Section 5: Monitoring & Maintenance Planning

Profile: `{PROFILE}`

### Key Questions Answered
**Q: How do you plan to monitor model performance and data quality in production?**  
**A:** We track **data health**, **feature drift** (7‑day vs 30‑day statistics), and **MAPE/RMSE/MAE** weekly.

**Q: What thresholds or indicators will prompt a retraining or update?**  
**A:** **15%+ drift** or **MAPE > 10%** for 2 consecutive weeks triggers action.

**Q: How will you handle data drift or pattern changes?**  
**A:** Investigate drivers (weather/events), adjust features/seasonality, and **retrain with recent data** if drift persists.

### Artifacts
- Monitoring snapshot: `{rel(plan_csv, BASE_DIR)}`  
- Alerts (if any): `{rel(alerts_csv, BASE_DIR)}`  
- Playbook: `{rel(playbook, BASE_DIR)}`

### Business Value (Executive View)
- **Early warning system** to prevent surprises.  
- **Clear thresholds** and **ownership**—so action is fast and consistent.
""")

# Consolidated upsert
def upsert_section(consolidated_path: Path, section_idx: int, title: str, new_block: str):
    if not consolidated_path.exists():
        consolidated_path.write_text("# SDS-CP036-powercast – Week 4 (Business Report)\n\n", encoding="utf-8")
    text = consolidated_path.read_text(encoding="utf-8")
    hdr_any = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*[1-5]\s*:\s*.*$")
    hdr_me = re.compile(fr"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*{section_idx}\s*:\s*{title}\s*$")
    m = hdr_me.search(text)
    if m:
        start = m.start()
        m2 = hdr_any.search(text, pos=m.end())
        end = m2.start() if m2 else len(text)
        text = (text[:start] + text[end:]).strip()
    text = text.rstrip() + "\n\n" + new_block.strip() + "\n"
    consolidated_path.write_text(text, encoding="utf-8")

upsert_section(CONSOLIDATED, 5, "Monitoring \& Maintenance Planning", s5_md.read_text(encoding="utf-8"))
print("✅ Upserted Section 5 into consolidated:", rel(CONSOLIDATED, BASE_DIR))


Wrote: /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk04_Section5_dev/reports/monitoring_playbook.md
Wrote: /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk04_Section5_dev/reports/SDS-CP036-powercast_Wk04_Section5_Business_Report.md
✅ Upserted Section 5 into consolidated: SDS-CP036-powercast_Wk04_Report_Business.md
